# B2-023 — Practice p19

*65 minutes.*

**Program layer:** Round 2 extension  
**Compute:** `compute.policy: cpu` · seed `20261015`  
**Qualified prerequisites:** `book1:F1-scientific-python`, `book1:F3-matrices`, `book1:F4-multivar-calculus`, `book1:F5-probability`, `book1:C1-ml-fundamentals`, `book1:C2-linear-models`, `book1:C5-neural-networks`, `book1:C6-pytorch`, `book1:C11-neural-training`, `B2-019-attention-transformers`, `B2-020-language-transformers`, `B2-021-cross-modal-transformers-vision`, `B2-022-probabilistic-latent-models`  
**Remediation links actually used:** [book1:F1-scientific-python](../../../../book1/units/F1-scientific-python/lesson.ipynb), [book1:F3-matrices](../../../../book1/units/F3-matrices/lesson.ipynb), [book1:F4-multivar-calculus](../../../../book1/units/F4-multivar-calculus/lesson.ipynb), [book1:F5-probability](../../../../book1/units/F5-probability/lesson.ipynb), [book1:C1-ml-fundamentals](../../../../book1/units/C1-ml-fundamentals/lesson.ipynb), [book1:C2-linear-models](../../../../book1/units/C2-linear-models/lesson.ipynb), [book1:C5-neural-networks](../../../../book1/units/C5-neural-networks/lesson.ipynb), [book1:C6-pytorch](../../../../book1/units/C6-pytorch/lesson.ipynb), [book1:C11-neural-training](../../../../book1/units/C11-neural-training/lesson.ipynb), [B2-019-attention-transformers](../../B2-019-attention-transformers/lesson.ipynb), [B2-020-language-transformers](../../B2-020-language-transformers/lesson.ipynb), [B2-021-cross-modal-transformers-vision](../../B2-021-cross-modal-transformers-vision/lesson.ipynb), [B2-022-probabilistic-latent-models](../../B2-022-probabilistic-latent-models/lesson.ipynb).

**Data boundary:** CPU only; use literal unit data, the seeded unit generator, and standard library/NumPy/Torch only. Do not use the web, downloads, external datasets, torchvision datasets, model hubs, pretrained models, checkpoints, or student data.

**Set:** C · **Type:** integrative · **Difficulty:** core · **Time budget:** 65 minutes  
**Concepts:** denoising-diffusion-probabilistic-models

## Task

Sample from the trained DDPM with fixed noise and certify the samples' statistics and mode coverage against held-out data.

**Training (rerun in this notebook).** Reproduce p18's protocol exactly: the same `train_rows()` / `heldout_rows()`, `make_schedule`, `q_sample`, `ddpm_loss`, `Denoiser`, `SCHEDULE = make_schedule(50, 1e-3, 0.2)`, `torch.manual_seed(SEED)` immediately before `model = Denoiser()`, the same Adam optimizer (`lr=1e-3`), and `train_ddpm(model, train_rows(), optimizer)` with its 1500 updates and its internal `torch.Generator().manual_seed(SEED)` drawing `t` then `eps`. Define `train_rows()` and `heldout_rows()` with no arguments; they return fresh float32 tensors `generative_data.train_tensor("mixture2d")` of shape `(256,2)` and `generative_data.heldout_tensor("mixture2d")` of shape `(64,2)`, built from the immutable `generative_data.TRAIN_IDS["mixture2d"]` and `generative_data.HELDOUT_IDS["mixture2d"]` in stored order. Held-out rows never enter a training call, a `backward()`, an optimizer step, or a hyperparameter choice. Clean training rows reach the model only through `q_sample`'s `x0`. Assert that the mean of the last 50 training losses is at most `0.60 *` the mean of the first 50.

**Sampler.** Implement `p_sample` and `sample_loop` with the p11 contract: `p_sample(model, x_t, t, z, schedule)` takes a Python int `t` in $1,\dots,T$ shared by the batch, calls `eps_hat = model(x_t, torch.full((B, 1), t / T, dtype=x_t.dtype))`, forms `mean = (x_t - beta[t] / sqrt(1 - alpha_bar[t]) * eps_hat) / sqrt(alpha[t])` with the coefficients taken as Python floats (`.item()`) from the float64 schedule, and returns `mean + sqrt(beta_tilde[t]) * z` for $t\ge2$ and exactly `mean` for $t=1$. `sample_loop(model, x_T, noises, schedule)` starts from `x = x_T` and, for `t = T, T-1, ..., 1` in that descending order, sets `x = p_sample(model, x, t, noises[t - 1], schedule)`; `noises` has shape `(T,B,d)`, `noises[0]` is ignored, and neither function draws random numbers or modifies its inputs in place.

**Sampling.** `sample_generator = torch.Generator().manual_seed(SEED + 2)`; draw `x_T = torch.randn(512, 2, generator=sample_generator)` and then `noises = torch.randn(50, 512, 2, generator=sample_generator)`. Under `torch.no_grad()`, `samples = sample_loop(model, x_T, noises, SCHEDULE)`. Let `H = heldout_rows()`, and define `inside_fraction(points)` as the fraction of rows within distance `0.6` of their nearest `mixture2d` centre (the p08 rule's radius test). Implement `mode_coverage` with the p08 contract.

**Certify.**
(A) `samples` has shape `(512,2)`, dtype float32, and is finite.
(B) **Mean:** every coordinate of `samples.mean(dim=0) - H.mean(dim=0)` is at most `0.35` in absolute value.
(C) **Covariance:** every entry of `torch.cov(samples.T) - torch.cov(H.T)` is at most `0.30` in absolute value.
(D) **Coverage:** `mode_coverage(samples, generative_data.mixture_centers(), 0.6)` covers all `4` components.
(E) **Precision:** `inside_fraction(samples) >= 0.75`, while the starting noise has `inside_fraction(x_T) <= 0.5`.
(F) **Determinism:** rerunning `sample_loop` with the same `x_T` and `noises` returns a `torch.equal` result.
(G) **Split certificate at the clean-row seam.** For the duration of the training call only, wrap the module-level `q_sample` (recording its `x0` argument) so that it records every row it receives (restore the original afterwards; held-out evaluation later legitimately uses the same seam). Then assert that no recorded row hashes, with `generative_data.row_sha256`, into the held-out part of `generative_data.ROW_SHA256["mixture2d"]` (the hashes at the positions in `generative_data.HELDOUT_IDS["mixture2d"]`), and that the recorded hashes include every training row's hash.
These thresholds were set from the frozen-seed reference run with margin; they are not a priori guarantees.

**Interpretation.** In four sentences, explain why (D) alone cannot distinguish samples from pure noise but (E) can, why the comparison uses held-out rather than training statistics, why the tolerances in (B) and (C) are as wide as they are given 64 held-out rows, and what a sampler that added noise at its final step would do to (E).

**Required answer-check assertions.** The training-decrease check and every item (A)–(G).

In [ ]:
import importlib.util
import math
from pathlib import Path

import torch
from torch import nn
import torch.nn.functional as F

SEED = 20261015
LOADER_CANDIDATES = (
    Path("../data/generative_data.py"),
    Path("units/B2-023-generative-models-diffusion/data/generative_data.py"),
)
loader_path = next(path for path in LOADER_CANDIDATES if path.is_file())
loader_spec = importlib.util.spec_from_file_location("generative_data", loader_path)
generative_data = importlib.util.module_from_spec(loader_spec)
loader_spec.loader.exec_module(generative_data)
torch.set_num_threads(1)

In [ ]:
# Write the required implementation, probes, and answer-check assertions here.

## Your response

Show every requested derivation, implementation, shape/dtype probe, training certificate, or audit. Use only the permitted literal data and the seeded unit generator, keep train and held-out roles separate, and end coding work with an answer-check section of executable assertions.